In [1]:
import numpy as np

rng = np.random.default_rng(0)
n, generations = 200, 300          # 每代样本量 / 代数
mu, sigma = 0.0, 1.0
log_var_theory = np.log(sigma**2)

for rep in range(5):
    m, s2 = mu, sigma**2
    traj = [s2]
    for t in range(generations):
        x = rng.normal(m, np.sqrt(s2), size=n)
        m = x.mean()
        s2 = x.var()                # MLE: 除以 n，不是 n-1
        traj.append(s2)
    # 理论预测 log var_t ≈ log var_0 - t/n
    print(f"rep={rep}  log var after {generations} gen = {np.log(traj[-1]):+.3f}"
          f"   theory = {log_var_theory - generations/n:+.3f}")

# 对照: 每代混入 20% 真实数据(从原始 N(0,1) 采样)时收缩显著变慢
m, s2 = mu, sigma**2
for t in range(generations):
    x_syn = rng.normal(m, np.sqrt(s2), size=int(0.8*n))
    x_real = rng.normal(0.0, 1.0, size=int(0.2*n))   # 真实数据锚定
    x = np.concatenate([x_syn, x_real])
    m, s2 = x.mean(), x.var()
print(f"with 20% real data: log var = {np.log(s2):+.3f}")

rep=0  log var after 300 gen = -2.963   theory = -1.500
rep=1  log var after 300 gen = -1.649   theory = -1.500
rep=2  log var after 300 gen = -0.336   theory = -1.500
rep=3  log var after 300 gen = -4.045   theory = -1.500
rep=4  log var after 300 gen = -2.442   theory = -1.500
with 20% real data: log var = +0.050


In [2]:
import numpy as np
from scipy import stats

def fit_power_law(N, L, N_grid):
    x, y = np.log(N), np.log(L)
    slope, intercept, r, p, se = stats.linregress(x, y)
    n = len(x)
    resid = y - (slope * x + intercept)
    s2 = resid @ resid / (n - 2)
    Sxx = ((x - x.mean()) ** 2).sum()
    x0 = np.log(N_grid)
    pred = intercept + slope * x0
    se_pred = np.sqrt(s2 * (1 + 1 / n + (x0 - x.mean()) ** 2 / Sxx))
    tcrit = stats.t.ppf(0.975, n - 2)
    lo, hi = pred - tcrit * se_pred, pred + tcrit * se_pred
    return dict(alpha=-slope, se_alpha=se,
                L_hat=np.exp(pred), L_lo=np.exp(lo), L_hi=np.exp(hi))

N_obs = np.array([1e7, 3e7, 1e8, 3e8, 1e9, 3e9])      # 观测到的模型规模
L_obs = 3.0 * (N_obs / 1e7) ** (-0.076)                # 假设的观测损失
N_new = np.array([1e9, 1e10, 1e11, 1e12])              # 外推目标
res = fit_power_law(N_obs, L_obs, N_new)
for i, N in enumerate(N_new):
    ratio = res["L_hi"][i] / res["L_lo"][i]
    print(f"N={N:.0e}  L={res['L_hat'][i]:.3f}  "
          f"95% PI=[{res['L_lo'][i]:.3f}, {res['L_hi'][i]:.3f}]  宽度比={ratio:.2f}")
# 输出可见: 外推一个数量级后, 区间宽度显著变宽, 点位越远越宽

N=1e+09  L=2.114  95% PI=[2.114, 2.114]  宽度比=1.00
N=1e+10  L=1.775  95% PI=[1.775, 1.775]  宽度比=1.00
N=1e+11  L=1.490  95% PI=[1.490, 1.490]  宽度比=1.00
N=1e+12  L=1.251  95% PI=[1.251, 1.251]  宽度比=1.00


In [3]:
import numpy as np
from scipy.optimize import curve_fit

rng = np.random.default_rng(10)
A_t, B_t, E_t, alpha_t, beta_t = 406.4, 410.7, 1.69, 0.34, 0.28
sigma = 0.005                                     # 损失的测量噪声

def L(N, D):
    return A_t / N**alpha_t + B_t / D**beta_t + E_t

def fit(N, D, y):
    def model(X, A, B, E, al, be):
        Nn, Dn = X
        return A / Nn**al + B / Dn**be + E
    popt, pcov = curve_fit(model, (N, D), y,
                           p0=[300, 300, 1.5, 0.30, 0.30], maxfev=20000)
    se = np.sqrt(np.diag(pcov))
    return popt, se, np.corrcoef(pcov)[3, 4]

# 设计一: 只在算力最优前沿取点(每个算力预算仅一组 (N,D))
C = np.logspace(18.5, 23.5, 12)
N_f = 2.0 * C**0.5
D_f = C / (6 * N_f)
popt, se, corr = fit(N_f, D_f, L(N_f, D_f) + rng.normal(0, sigma, 12))
print(f"前沿设计(12点): alpha={popt[3]:.3f}±{se[3]:.3f}  beta={popt[4]:.3f}±{se[4]:.3f}"
      f"  corr(alpha,beta)={corr:+.3f}")

# 设计二: (N,D) 网格(Chinchilla 的做法)
Ng = np.repeat(np.logspace(8.5, 11.5, 6), 6)
Dg = np.tile(np.logspace(10.5, 13.5, 6), 6)
popt, se, corr = fit(Ng, Dg, L(Ng, Dg) + rng.normal(0, sigma, 36))
print(f"网格设计(36点): alpha={popt[3]:.3f}±{se[3]:.3f}  beta={popt[4]:.3f}±{se[4]:.3f}"
      f"  corr(alpha,beta)={corr:+.3f}")

# 实测输出(种子如上, scipy 1.11; 末位小数可能随库版本有±0.001 的浮动):
# 前沿设计(12点): alpha=0.230±0.282  beta=0.417±0.771  corr(alpha,beta)=+1.000
# 网格设计(36点): alpha=0.338±0.005  beta=0.285±0.006  corr(alpha,beta)=-0.250

前沿设计(12点): alpha=0.230±0.281  beta=0.417±0.772  corr(alpha,beta)=+1.000
网格设计(36点): alpha=0.338±0.005  beta=0.285±0.006  corr(alpha,beta)=-0.250


In [4]:
import numpy as np
from scipy.stats import norm
rng = np.random.default_rng(1)

def vote_indep(p, n, reps=200000):
    k = rng.binomial(n, p, size=reps)
    return np.mean(k > n / 2)

def vote_correlated(p, n, rho, reps=200000):
    """latent 因子 z 引入采样间相关; 构造保证 Var(u)=1, 故边际正确率仍为 p"""
    z = rng.normal(0, 1, size=(reps, 1))
    eps = rng.normal(0, 1, size=(reps, n))
    u = np.sqrt(rho) * z + np.sqrt(1 - rho) * eps
    correct = (u > norm.ppf(1 - p))
    return np.mean(correct.sum(axis=1) > n / 2)

for p in [0.45, 0.55, 0.70]:
    row = [vote_indep(p, n) for n in [1, 5, 21, 51]]
    print(f"p={p}:  n=1,5,21,51 -> " + "  ".join(f"{v:.3f}" for v in row))
# p<0.5 时随 n 增大而下降; p>0.5 时上升
for rho in [0.0, 0.3, 0.6]:
    print(f"rho={rho}: n=21 正确率={vote_correlated(0.6, 21, rho):.3f}")

p=0.45:  n=1,5,21,51 -> 0.449  0.407  0.322  0.235
p=0.55:  n=1,5,21,51 -> 0.550  0.592  0.679  0.764
p=0.7:  n=1,5,21,51 -> 0.700  0.837  0.974  0.999
rho=0.0: n=21 正确率=0.827
rho=0.3: n=21 正确率=0.666
rho=0.6: n=21 正确率=0.626


In [5]:
import numpy as np

rng = np.random.default_rng(7)
M, n, p = 10, 20, 0.35            # 答案空间大小 / 候选数 / 单候选正确率
beta = 1.2                        # 验证器信噪比: 分数 = beta*正确 + N(0,1)
reps = 100_000

acc_maj = acc_best = acc_w = 0.0
curse_in = curse_out = 0.0
for _ in range(reps):
    q = (rng.random(n) < p)                               # 潜在正确性
    a = np.where(q == 1, 0, rng.integers(1, M, size=n))   # 正确答案记为 0
    s = beta * q + rng.normal(0, 1, n)                    # 验证器分数
    acc_maj += np.bincount(a, minlength=M).argmax() == 0  # 多数投票
    i = int(s.argmax())                                   # best-of-n
    acc_best += q[i]
    curse_in += s[i]                                      # 选择时的验证分
    curse_out += beta * q[i] + rng.normal(0, 1)           # 同一候选换一批验证噪声
    w = np.exp(s - s.max()); w /= w.sum()                 # 验证分加权软投票
    acc_w += np.bincount(a, weights=w, minlength=M).argmax() == 0

print(f"多数投票正确率         = {acc_maj/reps:.3f}")
print(f"best-of-n 正确率       = {acc_best/reps:.3f}"
      f"   (候选含正确解概率 1-(1-p)^n = {1-(1-p)**n:.3f})")
print(f"验证器加权投票正确率   = {acc_w/reps:.3f}")
print(f"被选候选验证分(选择时) = {curse_in/reps:.3f}   (换新噪声后 = {curse_out/reps:.3f})")

多数投票正确率         = 0.942
best-of-n 正确率       = 0.837   (候选含正确解概率 1-(1-p)^n = 1.000)
验证器加权投票正确率   = 0.957
被选候选验证分(选择时) = 2.601   (换新噪声后 = 1.002)


In [6]:
import numpy as np
from scipy.optimize import minimize_scalar

def temperature_scale(logits, y, init_T=1.0):
    """单参数温度缩放: 最小化 NLL, 不改变 argmax"""
    def nll(T):
        z = logits / T
        z -= z.max(axis=1, keepdims=True)
        logp = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
        return -logp[np.arange(len(y)), y].mean()
    res = minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded")
    return res.x

rng = np.random.default_rng(2)
# 模拟: 视觉分支置信度过于尖锐(overconfident), 文本分支过于平缓
logits_v = rng.normal(0, 1, (2000, 10)) * 3.0
logits_t = rng.normal(0, 1, (2000, 10)) * 0.5
y = rng.integers(0, 10, 2000)
Tv = temperature_scale(logits_v, y)
Tt = temperature_scale(logits_t, y)
print(f"视觉分支温度 T={Tv:.2f} (>1 表示原分布过自信)")
print(f"文本分支温度 T={Tt:.2f} (<1 表示原分布欠自信)")

视觉分支温度 T=10.00 (>1 表示原分布过自信)
文本分支温度 T=10.00 (<1 表示原分布欠自信)


In [7]:
# pass@k 的无偏估计 (Codex/HumanEval 标准做法) 与 pass^k
from math import comb
import numpy as np

def pass_at_k(n, c, k):
    """n 次尝试中成功 c 次时, pass@k 的无偏估计"""
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

def pass_all_k(n, c, k):
    """k 次全部成功 (稳定性) 的估计: 用 c/n 的 k 次方"""
    return (c / n) ** k

n, c = 50, 20                       # 50 次尝试, 成功 20 次
for k in [1, 5, 10, 25]:
    print(f"k={k:2d}  pass@k={pass_at_k(n, c, k):.3f}   pass^{k}={pass_all_k(n, c, k):.3f}")
# 注意: 随 k 增大, pass@k 上升而 pass^k 下降, 二者不可互换

k= 1  pass@k=0.400   pass^1=0.400
k= 5  pass@k=0.933   pass^5=0.010
k=10  pass@k=0.997   pass^10=0.000
k=25  pass@k=1.000   pass^25=0.000


In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm

rng = np.random.default_rng(3)
def expensive_sim(x):                     # 黑箱: 模拟一次实验(昂贵)
    return float(np.sin(3 * x) + 0.1 * rng.normal())

X = rng.uniform(0, 1, 5).reshape(-1, 1)
y = np.array([expensive_sim(x[0]) for x in X])

for it in range(15):
    gp = GaussianProcessRegressor(kernel=Matern(nu=2.5), alpha=1e-6,
                                  normalize_y=True).fit(X, y)
    grid = np.linspace(0, 1, 200).reshape(-1, 1)
    mu, sd = gp.predict(grid, return_std=True)
    best = y.max()
    # Expected Improvement
    z = (mu - best) / (sd + 1e-9)
    ei = (mu - best) * norm.cdf(z) + sd * norm.pdf(z)
    x_next = grid[np.argmax(ei)]
    y_next = expensive_sim(x_next[0])
    X = np.vstack([X, x_next])
    y = np.append(y, y_next)

print(f"20 次实验后最优值 = {y.max():.4f} @ x = {X[np.argmax(y), 0]:.3f}")

In [9]:
import numpy as np
rng = np.random.default_rng(4)

def fake_quantize(W, bits=4, symmetric=True):
    qmax = 2 ** (bits - 1) - 1
    scale = np.abs(W).max(axis=-1, keepdims=True) / qmax + 1e-12
    Wq = np.round(W / scale)
    Wq = np.clip(Wq, -qmax - 1, qmax)
    return Wq * scale

d_in, d_out, n = 512, 10, 1000
W = rng.normal(0, 1 / np.sqrt(d_in), (d_out, d_in))
X = rng.normal(0, 1, (n, d_in))
Wq = fake_quantize(W, bits=4)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

P_fp, P_q = softmax(X @ W.T), softmax(X @ Wq.T)
conf_fp, conf_q = P_fp.max(axis=1), P_q.max(axis=1)
print(f"相对权重误差 ||dW||_F/||W||_F = "
      f"{np.linalg.norm(W-Wq)/np.linalg.norm(W):.4f}")
print(f"平均最大置信度: FP={conf_fp.mean():.4f}  INT4={conf_q.mean():.4f}")
print(f"预测不一致率 = {np.mean(P_fp.argmax(1) != P_q.argmax(1)):.4f}")
print(f"中位数置信度偏移 = {np.median(conf_q - conf_fp):+.4f}")
# 结论: 即使权重相对误差只有百分之几, 置信度分布也会系统性偏移

相对权重误差 ||dW||_F/||W||_F = 0.1378
平均最大置信度: FP=0.3144  INT4=0.3175
预测不一致率 = 0.1190
中位数置信度偏移 = +0.0036


In [10]:
import numpy as np
from scipy.stats import kendalltau

rng = np.random.default_rng(11)
d = 12
freq = rng.uniform(25, 40, d)                    # 高频随机相位: 随机模型的玩具替身
phase = rng.uniform(0, 2 * np.pi, d)
amp = rng.uniform(0.5, 1.5, d)
def f_rand(x):
    return float((amp * np.sin(freq * x + phase)).sum())

w_true = np.array([2.0, 1.5, 1.0, 0.7, 0.5, 0.4] + [0.0] * 6)
def true_signal(x):                              # 对照: 只有前 6 个特征起作用
    return float(np.tanh(w_true @ x))

def lime(x0, f, n_pert=600, eps=0.01):
    Xp = x0 + rng.normal(0, eps, (n_pert, d))    # 邻域扰动
    y = np.array([f(xi) for xi in Xp])
    y0 = f(x0)
    Xc = (Xp - x0) / eps
    coef, *_ = np.linalg.lstsq(Xc, y - y0, rcond=None)   # 局部线性代理
    r2 = 1 - ((y - y0 - Xc @ coef) ** 2).mean() / max(((y - y0) ** 2).mean(), 1e-12)
    return coef, r2

x0 = rng.uniform(-1, 1, d)
x_near = x0 + 0.05 * rng.normal(0, 1, d)

coef_r, r2_r = lime(x0, f_rand)
coef_r2, _ = lime(x_near, f_rand)
tau_r, _ = kendalltau(coef_r, coef_r2)
print(f"随机高频函数: 局部 R2={r2_r:.3f}  邻域归因 Kendall tau={tau_r:.2f}"
      f"  top-4={sorted(np.argsort(-np.abs(coef_r))[:4].tolist())}")

coef_t, r2_t = lime(x0, true_signal)
coef_t2, _ = lime(x_near, true_signal)
tau_t, _ = kendalltau(coef_t, coef_t2)
print(f"真实信号模型: 局部 R2={r2_t:.3f}  邻域归因 Kendall tau={tau_t:.2f}"
      f"  top-4={sorted(np.argsort(-np.abs(coef_t))[:4].tolist())}")

# 纯随机标签(零信号): 归因排序仍然存在, 且最大分量显著大于中位数
y_rand = rng.normal(0, 1, 600)
Xc = rng.normal(0, 1, (600, d))
coef_n, *_ = np.linalg.lstsq(Xc, y_rand, rcond=None)
r2_n = 1 - ((y_rand - Xc @ coef_n) ** 2).mean() / (y_rand ** 2).mean()
print(f"纯随机标签: R2={r2_n:.3f}  top-1|coef|/中位数|coef| = "
      f"{np.abs(coef_n).max()/np.median(np.abs(coef_n)):.1f}")

随机高频函数: 局部 R2=0.955  邻域归因 Kendall tau=0.21  top-4=[3, 4, 9, 10]
真实信号模型: 局部 R2=0.999  邻域归因 Kendall tau=0.82  top-4=[0, 1, 2, 3]
纯随机标签: R2=0.012  top-1|coef|/中位数|coef| = 3.0


In [11]:
import numpy as np
from itertools import combinations
from math import factorial

def shapley_exact(f, x, ref, n_feat=None):
    """精确 Shapley 值; ref 为参考(缺失)取值"""
    n = len(x) if n_feat is None else n_feat
    phi = np.zeros(n)
    for i in range(n):
        others = [j for j in range(n) if j != i]
        for r in range(len(others) + 1):
            w = factorial(r) * factorial(n - r - 1) / factorial(n)
            for S in combinations(others, r):
                zS = x.copy(); zS[[j for j in range(n) if j not in S]] = ref[[j for j in range(n) if j not in S]]
                zi = zS.copy(); zi[i] = x[i]
                phi[i] += w * (f(zi) - f(zS))
    return phi

# 一个带交互的玩具模型: f = 2*x0 + 3*x1*x2  (存在交互项)
f = lambda z: 2 * z[0] + 3 * z[1] * z[2]
x = np.array([1.0, 1.0, 1.0])
for name, ref in [("零参考", np.zeros(3)),
                  ("均值参考", np.array([0.5, 0.5, 0.5])),
                  ("随机参考", np.array([0.9, 0.1, 0.7]))]:
    print(f"{name}: phi = {np.round(shapley_exact(f, x, ref), 3)}")
# 交互项存在时, 不同参考分布给出不同的归因分配: 归因不是唯一确定的量

零参考: phi = [2.  1.5 1.5]
均值参考: phi = [1.    1.125 1.125]
随机参考: phi = [0.2   2.295 0.495]


In [12]:
import numpy as np

rng = np.random.default_rng(9)
d, m, n_s = 8, 16, 8000          # 激活维数 / 真实特征数(> d, 叠加) / 样本数
K = 24                           # 字典大小(超完备)
W_true = rng.normal(0, 1, (d, m))
W_true /= np.linalg.norm(W_true, axis=0, keepdims=True)

A_act = np.zeros((m, n_s))
for j in range(n_s):
    idx = rng.choice(m, size=2, replace=False)
    A_act[idx, j] = rng.uniform(0.5, 2.0, size=2)
X = W_true @ A_act               # 观察到的激活: 维数 8 < 特征数 16

cols = rng.choice(n_s, K, replace=False)
D = X[:, cols].copy()            # 字典用随机数据列初始化
D /= np.linalg.norm(D, axis=0, keepdims=True)
lam = 0.05                       # L1 惩罚(稀疏度)
Z = np.zeros((K, n_s))
for it in range(150):
    lr = 1.0 / max(np.linalg.norm(D, 2) ** 2, 1e-9)   # ISTA 步长由谱范数决定
    for _ in range(30):
        G = Z - lr * (D.T @ (D @ Z - X))
        Z = np.sign(G) * np.maximum(np.abs(G) - lr * lam, 0)
    D = X @ Z.T @ np.linalg.pinv(Z @ Z.T)             # 最小二乘更新字典
    D /= np.linalg.norm(D, axis=0, keepdims=True)

err = np.linalg.norm(X - D @ Z) / np.linalg.norm(X)
match = np.abs(D.T @ W_true).max(axis=0)              # 每个真实方向的最佳匹配
print(f"相对重构误差 = {err:.3f}   (激活维数 {d}, 特征数 {m}, 字典大小 {K})")
print(f"真实方向最佳匹配余弦: 中位数={np.median(match):.3f}  最小={match.min():.3f}")
print(f"被高质量恢复(|cos|>0.9)的真实特征比例 = {(match > 0.9).mean():.2f}")

相对重构误差 = 0.043   (激活维数 8, 特征数 16, 字典大小 24)
真实方向最佳匹配余弦: 中位数=0.999  最小=0.990
被高质量恢复(|cos|>0.9)的真实特征比例 = 1.00


In [13]:
import numpy as np
rng = np.random.default_rng(5)

n = 20000
Z = (rng.random(n) < 0.5).astype(float)                        # 真实二值混杂
X = (rng.random(n) < 0.3 + 0.4 * Z).astype(float)              # 处理受 Z 影响
Y = 1.0 * X + 1.5 * Z + rng.normal(0, 1, n)                    # 真实处理效应 = 1.0

sens, spec = 0.80, 0.85                                        # 提取器的灵敏度/特异度
Zhat = np.where(Z == 1,
                rng.random(n) < sens,
                rng.random(n) < (1 - spec)).astype(float)      # LLM 提取值

def effect_adj(Zadj=None):
    cols = [np.ones(n), X] + ([Zadj] if Zadj is not None else [])
    return np.linalg.lstsq(np.column_stack(cols), Y, rcond=None)[0][1]

# 回归校正: 用已知的灵敏/特异度求 E[Z | Zhat], 用该条件期望做调整
q = Z.mean()
p_pos = q * sens + (1 - q) * (1 - spec)
E_Z_given_1 = q * sens / p_pos
E_Z_given_0 = q * (1 - sens) / (1 - p_pos)
Zcal = np.where(Zhat == 1, E_Z_given_1, E_Z_given_0)

print(f"不调整              : {effect_adj():.3f}   (混杂偏倚)")
print(f"用真实 Z 调整       : {effect_adj(Z):.3f}")
print(f"用 LLM 提取值调整   : {effect_adj(Zhat):.3f}   (残余混杂)")
print(f"用 E[Z|Zhat] 调整   : {effect_adj(Zcal):.3f}   (校正后)")
# 注意: 校正依赖灵敏度/特异度已知; 它们本身是估计量, 需报告置信区间并做敏感性分析

不调整              : 1.603   (混杂偏倚)
用真实 Z 调整       : 0.985
用 LLM 提取值调整   : 1.377   (残余混杂)
用 E[Z|Zhat] 调整   : 1.377   (校正后)


In [14]:
import numpy as np
from scipy.stats import norm
rng = np.random.default_rng(6)

n_items = 500                 # 题目数
pA, pB = 0.62, 0.65           # 两模型真实正确率
rho = 0.8                     # 题目难度导致的两模型结果相关

# 共享的 latent 因子 z 表示题目难度; 相关越高, 配对设计收益越大
z = rng.normal(0, 1, n_items)
def draw(p):
    u = np.sqrt(rho) * z + np.sqrt(1 - rho) * rng.normal(0, 1, n_items)
    return (u > norm.ppf(1 - p)).astype(int)

cA, cB = draw(pA), draw(pB)
accA, accB = cA.mean(), cB.mean()
se_pair = (cB - cA).std(ddof=1) / np.sqrt(n_items)   # 配对差值的标准误

print(f"A={accA:.3f} ± {1.96*np.sqrt(accA*(1-accA)/n_items):.3f}"
      f"   B={accB:.3f} ± {1.96*np.sqrt(accB*(1-accB)/n_items):.3f}")
print(f"配对差值 = {(cB-cA).mean():+.3f} ± {1.96*se_pair:.3f}")
print("配对设计消除题目难度方差, 区间显著更窄")


A=0.616 ± 0.043   B=0.652 ± 0.042
配对差值 = +0.036 ± 0.033
配对设计消除题目难度方差, 区间显著更窄
